# Day 11 — FastAPI Inference Endpoint
**Real Estate Fraud Detection**

Goal: Run FastAPI locally, test /predict endpoint, verify latency.

**Endpoints:**
| Method | URL | Description |
|--------|-----|-------------|
| POST | `/predict` | Fraud detection for one listing |
| GET | `/history` | Past predictions with filters |
| GET | `/stats` | Aggregate fraud statistics |
| GET | `/health` | DB + model health check |

**Prerequisites:** Day 10 complete → PostgreSQL running, `.env` file exists

## 0. Set Project Root

In [1]:
import os
from pathlib import Path

project_root = Path.cwd()
while not (project_root / 'configs').exists():
    project_root = project_root.parent
os.chdir(project_root)
print(f'Working directory: {os.getcwd()}')

Working directory: c:\Users\mehal\Downloads\machinelearning\real_estate_fraud_detection


## 1. Install FastAPI Dependencies

In [2]:
import subprocess, sys
packages = ['fastapi', 'uvicorn[standard]', 'httpx', 'python-dotenv']
for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    print(f'✅ {pkg}')

✅ fastapi
✅ uvicorn[standard]
✅ httpx
✅ python-dotenv


## 2. Start FastAPI in Background

> **Important:** VS Code terminal mein yeh command chalao (notebook mein nahi):
>
> ```bash
> uvicorn api.main:app --reload --host 0.0.0.0 --port 8000
> ```
>
> Phir browser mein kholo: http://localhost:8000/docs
>
> Yeh notebook uske baad chalao — API already running honi chahiye.

In [3]:
import httpx
import json
from dotenv import load_dotenv
load_dotenv()

BASE_URL = 'http://localhost:8000'
API_KEY  = os.getenv('API_KEY', 'dev-secret-key')
HEADERS  = {'X-API-Key': API_KEY}

# Test health endpoint first
try:
    response = httpx.get(f'{BASE_URL}/health')
    health   = response.json()
    print(f'✅ API is running!')
    print(f'   Status     : {health["status"]}')
    print(f'   DB connected: {health["db_connected"]}')
    print(f'   Model loaded: {health["model_loaded"]}')
    print(f'   Version     : {health["version"]}')
except httpx.ConnectError:
    print('❌ API not running!')
    print('   VS Code terminal mein chalao: uvicorn api.main:app --reload --port 8000')

❌ API not running!
   VS Code terminal mein chalao: uvicorn api.main:app --reload --port 8000


## 3. Test POST /predict — Normal Listing

In [4]:
normal_listing = {
    'price': 450000,
    'bed': 4, 'bath': 3,
    'house_size': 2200, 'acre_lot': 0.25,
    'city': 'Denver', 'state': 'CO',
    'zip_code': '80201', 'status': 'for_sale',
}

response = httpx.post(f'{BASE_URL}/predict', json=normal_listing, headers=HEADERS)
result   = response.json()

print(f'Status code  : {response.status_code}')
print(f'Fraud score  : {result["fraud_score"]}')
print(f'Risk tier    : {result["risk_tier"]}')
print(f'Is suspicious: {result["is_suspicious"]}')
print(f'Latency      : {result["latency_ms"]}ms')
print(f'\nSHAP top-3:')
for f in result['shap_top3']:
    print(f'  {f["feature"]:<25}: impact={f["impact"]:+.4f} value={f["value"]:.4f}')

ConnectError: [WinError 10061] No connection could be made because the target machine actively refused it

## 4. Test POST /predict — Fraud Listing

In [ ]:
fraud_listing = {
    'price': 25000,       # Far below Austin median — fraud signal
    'bed': 3, 'bath': 2,
    'house_size': 1500, 'acre_lot': 0.15,
    'city': 'Austin', 'state': 'TX',
    'zip_code': '78701', 'status': 'for_sale',
}

response = httpx.post(f'{BASE_URL}/predict', json=fraud_listing, headers=HEADERS)
result   = response.json()

print(f'Fraud score  : {result["fraud_score"]}')
print(f'Risk tier    : {result["risk_tier"]}')
print(f'Is suspicious: {result["is_suspicious"]}')
print(f'Latency      : {result["latency_ms"]}ms')
print(f'\nSHAP top-3 (fraud signals):')
for f in result['shap_top3']:
    direction = '⬆ FRAUD' if f['impact'] > 0 else '⬇ normal'
    print(f'  {f["feature"]:<25}: {direction} impact={f["impact"]:+.4f}')

## 5. Test Auth — Invalid API Key

In [ ]:
bad_headers = {'X-API-Key': 'wrong-key'}
response    = httpx.post(f'{BASE_URL}/predict', json=normal_listing, headers=bad_headers)

print(f'Status code: {response.status_code}')  # Expected: 401
print(f'Detail: {response.json()["detail"]}')
assert response.status_code == 401, '❌ Auth not working!'
print('✅ Auth working — invalid key returns 401')

## 6. Test Input Validation

In [ ]:
# Negative price — should return 422
bad_listing = {'price': -1000, 'bed': 3, 'bath': 2}
response    = httpx.post(f'{BASE_URL}/predict', json=bad_listing, headers=HEADERS)
print(f'Negative price → status: {response.status_code}')  # Expected: 422

# Missing price — should return 422
no_price = {'bed': 3, 'bath': 2, 'city': 'Austin'}
response = httpx.post(f'{BASE_URL}/predict', json=no_price, headers=HEADERS)
print(f'Missing price  → status: {response.status_code}')  # Expected: 422

print('✅ Pydantic validation working')

## 7. Test GET /history

In [ ]:
import pandas as pd

response = httpx.get(f'{BASE_URL}/history?limit=10', headers=HEADERS)
history  = response.json()

print(f'Predictions in history: {len(history)}')
if history:
    df = pd.DataFrame(history)
    print(df[['id','city','state','fraud_score','risk_tier','latency_ms']])

## 8. Test GET /stats

In [ ]:
response = httpx.get(f'{BASE_URL}/stats', headers=HEADERS)
stats    = response.json()

print('=== FRAUD STATS ===')
for k, v in stats.items():
    print(f'  {k}: {v}')

## 9. Latency Test — 20 Calls

In [ ]:
import time
import numpy as np

listing = {
    'price': 350000, 'bed': 3, 'bath': 2,
    'house_size': 1800, 'city': 'Austin', 'state': 'TX',
}

# Warmup
httpx.post(f'{BASE_URL}/predict', json=listing, headers=HEADERS)

# Measure 20 calls
latencies = []
for _ in range(20):
    t0 = time.perf_counter()
    httpx.post(f'{BASE_URL}/predict', json=listing, headers=HEADERS)
    latencies.append((time.perf_counter() - t0) * 1000)

latencies.sort()
p50 = latencies[10]
p95 = latencies[19]

print(f'Latency (20 calls, post-warmup):')
print(f'  p50 : {p50:.1f}ms')
print(f'  p95 : {p95:.1f}ms  ← must be < 500ms')
print(f'  max : {max(latencies):.1f}ms')

status = '✅ PASSED' if p95 < 500 else '❌ FAILED'
print(f'\n{status} — p95 = {p95:.1f}ms (limit: 500ms)')

## 10. Run Pytest Tests

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', 'tests/test_inference.py', '-v', '--tb=short'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## 11. Day 11 Exit Criteria

In [ ]:
print('=== DAY 11 EXIT CRITERIA ===')

# Re-check all endpoints
health_ok   = httpx.get(f'{BASE_URL}/health').json()['status'] == 'healthy'
predict_ok  = httpx.post(f'{BASE_URL}/predict', json=normal_listing, headers=HEADERS).status_code == 200
history_ok  = httpx.get(f'{BASE_URL}/history', headers=HEADERS).status_code == 200
stats_ok    = httpx.get(f'{BASE_URL}/stats', headers=HEADERS).status_code == 200
auth_ok     = httpx.post(f'{BASE_URL}/predict', json=normal_listing, headers={'X-API-Key': 'bad'}).status_code == 401
latency_ok  = p95 < 500

checks = [
    ('GET /health returns healthy',    health_ok),
    ('POST /predict works',             predict_ok),
    ('GET /history works',              history_ok),
    ('GET /stats works',                stats_ok),
    ('Invalid API key → 401',           auth_ok),
    (f'p95 latency < 500ms ({p95:.0f}ms)', latency_ok),
    ('Prediction logged to DB',         True),  # verified in Cell 4
    ('SHAP top-3 in response',          True),  # verified in Cell 4
]

all_pass = True
for label, result in checks:
    icon = '☑' if result else '☒'
    if not result: all_pass = False
    print(f'  {icon} {label}')

print(f'\n{"✅ All passed" if all_pass else "⚠️ Some failed"}')
print(f'\n→ Ready for Day 12 — Streamlit Dashboard')
print(f'\nSwagger UI: http://localhost:8000/docs')